# IOAI — 2024 Mock Competition Pendulum Motion (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/jaredliw/ioai-tsp-2025"
CLONE = "ioai-tsp-2025"
SUBDIR = "noai-china-2024/pendulum-motion"
WORKDIR = "noai-china-2024/pendulum-motion"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Solving Pendulum Motion with Missing Data

> **NOAI China 2024 — Question 4** (APOAI 2025 Mock, Q4).

A damped pendulum is recorded as `(t, theta)` but the sensor cut out for a few seconds, and
during that gap a constant downward force `F` started acting. Recover the physics parameters
from the data alone:

1. rope length `l`,  2. air-resistance `mu`,  3. external force `F`,
4. next time `theta=0` after recording ends `t_nextzerotheta`,  5. force-onset time `t_Fput`.

Equation of motion (m=1, g=9.8): $l\,\ddot\theta = -g\sin\theta - \mu l\,\dot\theta - F_d$, where $F_d = F\sin\theta$ for $t \ge t_{Fput}$ else 0.

The official A/B sets are hidden and have **different** parameters, so the deliverable is a
*method*. Here we solve on `pendulum_train.csv`, write `submission_train.csv`, and self-check
by reconstructing $\theta(t)$ with `solve_ivp` and comparing to the data.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
m, g = 1.0, 9.8

## Load data and locate the sensor gap

In [ ]:
df = pd.read_csv('pendulum_train.csv')

def get_gap(d):
    i = d['t'].diff().argmax()
    return d['t'].iloc[i - 1], d['t'].iloc[i]

gap_start, gap_end = get_gap(df)
print('gap:', gap_start, '->', gap_end)
plt.scatter(df['t'], df['theta'], s=6); plt.xlabel('t'); plt.ylabel('theta');

## Estimate derivatives (central differences) and assemble training tensor

In [ ]:
def to_tensor(d):
    t = torch.tensor(d['t'].to_numpy())
    th = torch.tensor(d['theta'].to_numpy())
    dth = torch.gradient(th, spacing=(t[1] - t[0]))[0]
    d2th = torch.gradient(dth, spacing=(t[1] - t[0]))[0]
    return torch.vstack([t, th, dth, d2th]).T

def preprocess(d):
    gs, ge = get_gap(d)
    first = to_tensor(d[d['t'] <= gs]); first[0, 2:] = 0
    second = to_tensor(d[d['t'] >= ge])[1:]   # first row after gap has no reliable derivative
    return torch.cat([first, second])

data = preprocess(df)
data.shape

## Regress `l`, `mu`, `F` by driving the ODE residual to zero

We fix `t_Fput` to the gap midpoint while fitting (we lack data inside the gap), then grid-search it.

In [ ]:
class MyModel(nn.Module):
    def __init__(self, gap_t):
        super().__init__()
        self.gap_t = gap_t
        self.raw_l = nn.Parameter(torch.randn(()))
        self.raw_mu = nn.Parameter(torch.randn(()))
        self.raw_F = nn.Parameter(torch.randn(()))
    @property
    def l(self):  return F.softplus(self.raw_l)
    @property
    def mu(self): return F.softplus(self.raw_mu)
    @property
    def Fmag(self): return F.softplus(self.raw_F)
    def forward(self, x):
        t, th, dth, d2th = x.unbind(dim=1)
        Fd = self.Fmag * torch.sin(th) * (t >= self.gap_t).float()
        # residual of  m l th'' + m g sin th + mu l th' + Fd  = 0
        return m * self.l * d2th + m * g * torch.sin(th) + self.mu * self.l * dth + Fd

model = MyModel((gap_start + gap_end) / 2)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for epoch in range(8000):
    res = model(data).squeeze(-1)
    loss = F.mse_loss(res, torch.zeros_like(res))
    opt.zero_grad(); loss.backward(); opt.step()
    if (epoch + 1) % 2000 == 0:
        print(f'epoch {epoch+1}  loss {loss.item():.6e}')
l, mu, Fmag = model.l.item(), model.mu.item(), model.Fmag.item()
print(f'l={l:.4f}  mu={mu:.4f}  F={Fmag:.4f}')

## Find `t_Fput` and next `theta=0` by reconstructing the trajectory

In [ ]:
theta0 = df[df['t'] == 0]['theta'].item()
t_end = df['t'].max()

def ivp(t_F, t_eval):
    def ode(t, y):
        th, dth = y
        Fd = (Fmag / (m * l)) * np.sin(th) if t >= t_F else 0.0
        return [dth, -(g / l) * np.sin(th) - (mu / m) * dth - Fd]
    return solve_ivp(ode, (0, t_eval.max()), (theta0, 0), t_eval=t_eval).y[0]

# grid-search t_Fput over the gap (loss jumps sharply, so grid beats gradient methods)
obs_t = df['t'].to_numpy(); obs_th = df['theta'].to_numpy()
cands = np.arange(gap_start, gap_end, 0.005)
t_Fput = min(cands, key=lambda tf: np.mean((ivp(tf, obs_t) - obs_th) ** 2))
print('t_Fput =', round(t_Fput, 4))

In [ ]:
ts = np.arange(0, t_end + 5, 0.005)
th_pred = ivp(t_Fput, ts)
after = ts > t_end
sign_change = np.where(np.diff(np.sign(th_pred[after])) != 0)[0][0]
ts_after = ts[after]
t_nextzerotheta = (ts_after[sign_change] + ts_after[sign_change + 1]) / 2
print('t_nextzerotheta =', round(t_nextzerotheta, 4))

plt.scatter(df['t'], df['theta'], s=6, label='data')
plt.plot(ts, th_pred, color='C1', label='reconstructed')
plt.axvline(t_Fput, ls='--', c='gray'); plt.legend(); plt.xlabel('t'); plt.ylabel('theta');

## Self-check + write `submission_train.csv`

Reconstruction MSE over the recorded window measures how well the recovered parameters explain the data.

In [ ]:
mse = float(np.mean((ivp(t_Fput, obs_t) - obs_th) ** 2))
print(f'Reconstruction MSE on recorded data: {mse:.6e}  (lower is better)')

pd.DataFrame({'l': l, 'miu': mu, 'F': Fmag,
              't_nextzerotheta': t_nextzerotheta, 't_Fput': t_Fput},
             index=[0]).to_csv('submission_train.csv', index=False)
print('wrote submission_train.csv')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_train.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)